# 🚀 AIC 2026 - Master Pipeline

Notebook này chứa toàn bộ luồng chạy hoàn chỉnh của hệ thống tìm kiếm video (Ensemble Zero-Shot + Projection Head).

### Bước 0: Kiểm tra hệ thống GPU

In [18]:
!python 0_check_system.py

=== KIỂM TRA HỆ THỐNG ===
PyTorch Version: 2.4.1+cpu
CUDA Available: False
⚠️ DANG CHAY TREN CPU - Chua nhan GPU!


### Bước 0.5: Khởi tạo Dữ liệu BTC (Tạo metadata)
Đọc thư mục chứa ảnh `data/keyframes/` để tạo ra danh sách và index tất cả các ảnh (`metadata.jsonl`). Nếu bạn đã làm rồi thì có thể bỏ qua, nếu muốn format lại thì thêm cờ `--force`.

In [19]:
!python scripts/import_btc_data.py

2026-08-15 20:31:11,562 - INFO - Scanned keyframes for 29 videos, total 22248 frames.
2026-08-15 20:31:11,564 - INFO - [L21_V001] Metadata JSONL đã tồn tại hợp lệ. Skipping (dùng --force để ghi đè).
2026-08-15 20:31:11,574 - INFO - [L21_V002] Metadata JSONL đã tồn tại hợp lệ. Skipping (dùng --force để ghi đè).
2026-08-15 20:31:11,584 - INFO - [L21_V003] Metadata JSONL đã tồn tại hợp lệ. Skipping (dùng --force để ghi đè).
2026-08-15 20:31:11,592 - INFO - [L21_V005] Metadata JSONL đã tồn tại hợp lệ. Skipping (dùng --force để ghi đè).
2026-08-15 20:31:11,600 - INFO - [L21_V006] Metadata JSONL đã tồn tại hợp lệ. Skipping (dùng --force để ghi đè).
2026-08-15 20:31:11,701 - INFO - [L21_V007] Metadata JSONL đã tồn tại hợp lệ. Skipping (dùng --force để ghi đè).
2026-08-15 20:31:11,707 - INFO - [L21_V008] Metadata JSONL đã tồn tại hợp lệ. Skipping (dùng --force để ghi đè).
2026-08-15 20:31:11,719 - INFO - [L21_V009] Metadata JSONL đã tồn tại hợp lệ. Skipping (dùng --force để ghi đè).
2026-08-15

### Bước 1: Trích xuất Đặc trưng (Ensemble 3 Mô hình)
Chạy qua toàn bộ dữ liệu ảnh đã lấy từ bước 0.5 và lưu dưới dạng `.npy`, đồng thời tạo FAISS index tạm thời (2304 chiều).

In [23]:
!python 1_extract_and_build_index.py

I0000 00:00:1786801143.668913   21340 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786801148.583069   21340 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-15 20:39:13,400 - INFO - === HỆ THỐNG SEARCH AIC 2026 (ENSEMBLE ZERO-SHOT) ===
2026-08-15 20:39:13,401 - INFO - === BƯỚC 1: TRÍCH XUẤT ĐẶC TRƯNG ENSEMBLE (3 MODELS) ===
2026-08-15 20:39:13,402 - INFO - -> Nạp đồng thời 3 mô hình (CLIP, BLIP Base, BEiT v1) vào VRAM...
2026-08-15 20:39:13,402 - INFO - Parsing model identifier. Schema: None, Identifier: ViT-L-14
2026-08-15 20:39:13,403 - INFO - Loaded built-in ViT-L-14 model conf

### Bước 2: Huấn luyện Mạng thần kinh (Projection Head)
Dùng các file đặc trưng đã trích xuất ở Bước 1 kết hợp với file `captions_dummy.json` (hoặc nhãn từ BTC) để dạy cho máy học cách khớp câu tiếng Việt vào hình ảnh.

Bạn có thể đổi số `--epochs 50` thành số vòng lặp mà bạn muốn.

In [24]:
!python 2_train_projection_head.py --epochs 50

2026-08-15 20:44:53,059 - INFO - Đang nạp mô hình ngôn ngữ CLIP (FP16) để làm giám khảo (đóng băng)...
2026-08-15 20:44:53,059 - INFO - Parsing model identifier. Schema: None, Identifier: ViT-L-14
2026-08-15 20:44:53,059 - INFO - Loaded built-in ViT-L-14 model config.
2026-08-15 20:44:54,035 - INFO - Instantiating model architecture: CLIP
2026-08-15 20:45:01,819 - INFO - Loading full pretrained weights from: C:\Users\Administrator\.cache\huggingface\hub\models--laion--CLIP-ViT-L-14-laion2B-s32B-b82K\snapshots\1627032197142fbe2a7cfec626f4ced3ae60d07a\open_clip_pytorch_model.bin
2026-08-15 20:45:04,736 - INFO - Final image preprocessing configuration set: {'size': (224, 224), 'mode': 'RGB', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'interpolation': 'bicubic', 'resize_mode': 'shortest', 'fill_color': 0}
2026-08-15 20:45:04,737 - INFO - Model ViT-L-14 creation process complete.
2026-08-15 20:45:04,737 - INFO - Parsing tokenizer identifier. Schema: None, Identifier: ViT-L-14
2026-08-

### Bước 3: Rebuild FAISS Index
Sau khi mô hình Projection Head `projection_head_latest.pth` được lưu ở Bước 2. Ta sẽ chạy toàn bộ vector ảnh (2304d) qua mô hình này để ép về chuẩn 768d của văn bản, rồi lưu lại thành FAISS Index mới.

In [27]:
!python 3_rebuild_projected_index.py

2026-08-15 20:48:10,458 - INFO - === BƯỚC 3: XÂY DỰNG LẠI FAISS INDEX VỚI PROJECTION HEAD ===
2026-08-15 20:48:10,611 - INFO - Phát hiện số chiều của Raw Vector: 2304
2026-08-15 20:48:10,935 - INFO - Đã nạp Projection Head thành công.
2026-08-15 20:49:13,069 - INFO - Đã xây dựng xong FAISS Index (Projected 768 chiều) với 17561 keyframes.


### Bước 4: Khởi động Backend API / Hoặc Query Trực Tiếp
Sau khi có FAISS Index hoàn chỉnh, bạn có thể chạy API để Web App Frontend kết nối tới.

In [ ]:
!python main.py

### (Tùy chọn) Chạy Test Truy vấn Trực tiếp trên Notebook

In [28]:
import sys
import os
from backend.embedding.search_engine import VectorSearchEngine

engine = VectorSearchEngine()
engine.load_index()

query = "một bức ảnh về chiếc xe màu đỏ"
results = engine.search_single(query, top_k=5)

print(f"Kết quả cho truy vấn: {results['query_vi']}")
for r in results['results']:
    print(f"- Video ID: {r['video_id']} | Frame: {r['frame_id']} | Score: {r['score']:.4f}")

c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-15 20:50:09,838 - INFO - Đã nạp thành công FAISS Index (768d, 17561 items).
2026-08-15 20:50:09,840 - INFO - Parsing model identifier. Schema: None, Identifier: ViT-L-14
2026-08-15 20:50:09,840 - INFO - Loaded built-in ViT-L-14 model config.
2026-08-15 20:50:10,942 - INFO - Instantiating model architecture: CLIP
2026-08-15 20:50:18,103 - INFO - Loading full pretrained weights from: C:\Users\Administrator\.cache\huggingface\hub\models--laion--CLIP-ViT-L-14-laion2B-s32B-b82K\snapshots\1627032197142fbe2a7cfec626f4ced3ae60d07a\open_clip_pytorch_model.bin
2026-08-15 20:50:21,102 - INFO - Final image preprocessing configuration set: {'size': (224, 224), 'mode': 'RGB', 'mean': (0.5, 0.5, 0.5), 'std': (0.

Kết quả cho truy vấn: một bức ảnh về chiếc xe màu đỏ
- Video ID: L21_V002 | Frame: 12 | Score: 0.0885
- Video ID: L21_V001 | Frame: 12 | Score: 0.0805
- Video ID: L21_V001 | Frame: 0 | Score: 0.0726
- Video ID: L21_V002 | Frame: 0 | Score: 0.0724
- Video ID: L21_V009 | Frame: 0 | Score: 0.0721
